# Teaching NanoGPT to Do Math - DPO Fine-tuning
# SC3000/CZ3005 Assignment 1
## Team Members: [List your team members and contributions here]
## - Chan Zi Jian - Task 1
## - Grover Ekhnoor Kaur - Task 2 and 3 
## - Irani Tanya - Task 3 and 4

## Project Overview
This notebook implements Direct Preference Optimization (DPO) to fine-tune a pretrained 
NanoGPT model to solve arithmetic and algebra problems. The model was originally trained 
on general QA data and lacks mathematical reasoning abilities.

### What is DPO?
DPO (Direct Preference Optimization) is a reinforcement learning technique that:
1. Takes pairs of (negative, positive) responses to the same prompt
2. Trains the model to prefer positive responses over negative ones
3. Uses a mathematical loss function to maximize the probability of good responses

### Our Approach
1. Generate negative samples using the base model (incorrect/uninformative answers)
2. Create positive samples with correct, well-explained math solutions
3. Train the model to learn the difference between good and bad responses
4. Test on arithmetic and algebra problems

### Step 1: Install necesscary packages
Why these packages?
- matplotlib: For visualization and plotting training metrics
- torch: PyTorch deep learning framework for neural network operations
- transformers: Hugging Face library (though not directly used here)
- datasets: For data loading utilities
- tiktoken: Tokenization library
- wandb: Weights & Biases for experiment tracking (optional)
- tqdm: Progress bars for training loops

In [ ]:
!pip install matplotlib
!pip install torch numpy transformers datasets tiktoken wandb tqdm
!pip uninstall -y torch torchvision torchaudio #i did this so that we can access the gpu, so um dont run this , if do then restart session and dont run it after that
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124


Found existing installation: torch 2.8.0+cu126
Uninstalling torch-2.8.0+cu126:
  Successfully uninstalled torch-2.8.0+cu126
Found existing installation: torchvision 0.23.0+cu126
Uninstalling torchvision-0.23.0+cu126:
  Successfully uninstalled torchvision-0.23.0+cu126
Found existing installation: torchaudio 2.8.0+cu126
Uninstalling torchaudio-2.8.0+cu126:
  Successfully uninstalled torchaudio-2.8.0+cu126
Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 107.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 66.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 93.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device name:", torch.cuda.get_device_name(0))


CUDA available: True
Device name: Tesla T4


### Step 2: Package imports and configuration
Configuration Explanation:
- beta (1.00): Controls how strongly we prefer positive over negative samples
  Higher beta = stronger preference learning
- device: Automatically use GPU if available, otherwise CPU
- base_lr (1e-4): Learning rate - how fast the model updates weights
- epochs (6): Number of complete passes through the training data
- batch_size (64): Number of samples processed together (affects memory usage)
- max_length (64): Maximum sequence length for input text
- temperature (0.8): Controls randomness in generation (lower = more deterministic)
- top_k (200): Only sample from top 200 most likely next tokens

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import pickle
from model import GPT, GPTConfig
import random
from tqdm import tqdm
import time
import json
import matplotlib.pyplot as plt
from torch.cuda.amp import GradScaler, autocast
# Configuration
beta = 1.00
device = 'cuda' if torch.cuda.is_available() else 'cpu'
base_lr = 1e-4
epochs = 6
batch_size = 64
max_length =64
num_samples = 1
max_new_tokens = 200
temperature = 0.8
top_k = 200
# tokenizer
with open("meta.pkl", "rb") as f:
    meta = pickle.load(f)
stoi, itos = meta["stoi"], meta["itos"]
def encode(s): return [stoi[c] for c in s]
def decode(l): return ''.join([itos[i] for i in l])

In [ ]:
!nvidia-smi


Wed Oct 22 07:35:46 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device name:", torch.cuda.get_device_name(0))

CUDA available: True
Device name: Tesla T4


### Step 3: Define helper functions

In [ ]:
def compute_logprob(input_ids):
    inputs = input_ids[:, :-1]
    targets = input_ids[:, 1:]
    logits, _ = gpt(inputs, full_seq=True)
    B, T, V = logits.size()
    logits_flat = logits.reshape(-1, V)
    targets_flat = targets.reshape(-1)
    loss = F.cross_entropy(logits_flat, targets_flat, ignore_index=0, reduction='none')
    loss = loss.reshape(B, T)
    attention_mask = (targets != 0).float()
    loss = (loss * attention_mask).sum(dim=1) / attention_mask.sum(dim=1)
    return -loss

def pad_or_truncate(seq, max_length):
    return seq[-max_length:] if len(seq) > max_length else seq + [0] * (max_length - len(seq))

def get_batches(lines, batch_size):
    random.shuffle(lines)
    #for l in lines:
    #    print(l[1])
    for i in range(0, len(lines), batch_size):
        batch = lines[i:i+batch_size]
        if len(batch) < batch_size:
            continue
        neg_inputs = [pad_or_truncate(encode(p['negative'] + '\n\n\n\n'), max_length) for p in batch]
        pos_inputs = [pad_or_truncate(encode(p['positive'] + '\n\n\n\n'), max_length) for p in batch]
        neg_tensor = torch.tensor(neg_inputs, dtype=torch.long, device=device)
        pos_tensor = torch.tensor(pos_inputs, dtype=torch.long, device=device)
        yield neg_tensor, pos_tensor

### Step 4: Load the pretrained NanoGPT model

In [ ]:
ckpt = torch.load("gpt.pt", map_location=device)
gptconf = GPTConfig(**ckpt['model_args'])
gpt = GPT(gptconf)
state_dict = ckpt['model']
unwanted_prefix = '_orig_mod.'
for k in list(state_dict.keys()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
gpt.to(device).train()

GPT(
  (transformer): ModuleDict(
    (wte): Embedding(74, 348)
    (wpe): Embedding(256, 348)
    (drop): Dropout(p=0.2, inplace=False)
    (h): ModuleList(
      (0-5): 6 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=348, out_features=1044, bias=False)
          (c_proj): Linear(in_features=348, out_features=348, bias=False)
          (attn_dropout): Dropout(p=0.2, inplace=False)
          (resid_dropout): Dropout(p=0.2, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=348, out_features=1392, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=1392, out_features=348, bias=False)
          (dropout): Dropout(p=0.2, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=348, out_features=74, bias=False)
)

### Step 5: Load Data (**students are required to complete this part!**)
Data Loading and Cleaning:

Task 1 Implementation - Building Positive-Negative Pairs:
We load data pairs where:
- Negative: Model's incorrect/unhelpful response (e.g., "Sorry, I don't know")
- Positive: Correct, well-explained solution with reasoning

Data Cleaning Process:
1. Load JSON file containing pos-neg pairs
2. Filter out characters not in vocabulary (prevents encoding errors)
3. Verify data structure and size

Why cleaning is important:
- Unknown characters cause tokenization errors
- Clean data ensures smooth training
- Maintains consistency with pretrained vocabulary

In [ ]:
# Load the data from pos_neg_pairs.json
with open("./pos_neg_pairs.json", "r") as f:
    lines = json.load(f)

# Load vocabulary to filter characters
import pickle
with open("meta.pkl", "rb") as f:
    meta = pickle.load(f)
stoi_check = meta["stoi"]
valid_chars = set(stoi_check.keys())

# Clean function to remove unknown characters
def clean_text(text):
    return ''.join(c for c in text if c in valid_chars)

# Clean all the data
for pair in lines:
    pair['negative'] = clean_text(pair['negative'])
    pair['positive'] = clean_text(pair['positive'])

# Check if the data loaded correctly
print(f"Loaded {len(lines)} positive/negative pairs")
print(f"Sample data structure:")
print(f"  Negative: {lines[0]['negative']}")
print(f"  Positive: {lines[0]['positive']}")

Loaded 100000 positive/negative pairs
Sample data structure:
  Negative: 2496/96=? Sorry, I don't know
  Positive: 2496/96=? The answer is 26 because 2496 / 96 equals 26.


In [ ]:
# Load data from ./data/pos_neg_pairs.json
'''with open("./pos_neg_pairs.json", "r") as f:
    lines = json.load(f)

# Check Data loaded correctly
print(f"Loaded {len(lines)} positive/negative pairs")
print(f"Sample data structure:")
print(f"  Negative: {lines[0]['negative']}")
print(f"  Positive: {lines[0]['positive']}")'''

Loaded 100000 positive/negative pairs
Sample data structure:
  Negative: 2496/96=? Sorry, I don't know!
  Positive: 2496/96=? The answer is 26 because 2496 / 96 equals 26.


### Step 6: Build the optimizer and scheduler (**students are required to complete this part!**)
Optimizer and Scheduler Setup:

1. AdamW Optimizer:
   - Adaptive learning rate for each parameter
   - Weight decay (0.01) prevents overfitting
   - Well-suited for transformer models

2. Cosine Annealing Scheduler:
   - Learning rate starts at base_lr
   - Gradually decreases following cosine curve
   - Ends at 10% of base_lr (eta_min)
   - Helps model converge to better solution
   
Why this combination?
- AdamW handles different parameter scales well
- Cosine annealing provides smooth learning rate decay
- Together they provide stable, effective training


In [ ]:
# recommend to use the AdamW optimizer
# AdamW optimizer
optimizer = torch.optim.AdamW(gpt.parameters(), lr=base_lr, weight_decay=0.01)

# Optional: Learning rate scheduler (cosine annealing)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=epochs * (len(lines) // batch_size),
    eta_min=base_lr * 0.1
)

### Step 7: Begin training (**students are required to complete this part!**)
Direct Preference Optimization (DPO) Training - Simplified Explanation:

Think of DPO like teaching a student by showing them right and wrong answers:
- Show the model a BAD answer (negative): "Sorry, I don't know"
- Show the model a GOOD answer (positive): "The answer is 6 because 48/8 equals 6"
- Train the model to prefer the good answer over the bad one

How we do this:
1. Calculate how "likely" the model thinks each answer is (log probability)
   - Higher probability = model thinks this answer is more likely
   
2. Compare them: How much better is the good answer than the bad one?
   - If good answer has much higher probability → model is learning well ✓
   - If both have similar probability → model needs more training ✗

3. The Loss Function (what we're trying to minimize):
   loss = -log(sigmoid(beta * (good_prob - bad_prob))) - 0.1 * good_prob
   
   Breaking this down:
   - (good_prob - bad_prob): Difference between good and bad answers
   - beta: Multiplier to make the difference matter more (set to 1.0)
   - sigmoid(...): Squashes result between 0 and 1
   - -log(...): Converts to a loss (lower is better)
   - -0.1 * good_prob: Small bonus to also learn the good answer itself
   
   In plain English: "Punish the model if it doesn't strongly prefer good over bad"

Training Loop (what happens each iteration):
1. Take a batch of (bad, good) answer pairs
2. Ask model: "How likely is each answer?"
3. Calculate loss: "How much does model prefer good over bad?"
4. Update model to increase preference for good answers
5. Repeat with next batch

After many iterations:
- Model learns to generate good answers (with explanations)
- Model avoids generating bad answers ("I don't know")

In [ ]:
from torch.amp import autocast, GradScaler
from tqdm import tqdm

# 1. Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
gpt = GPT(GPTConfig()).to(device)

# 2. Use your optimizer and scheduler
optimizer = torch.optim.AdamW(gpt.parameters(), lr=base_lr, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=epochs * (len(lines) // batch_size),
    eta_min=base_lr * 0.1
)

# 3. Mixed precision scaler
scaler = GradScaler()

# 4. Save path
save_path = "./dpo.pt"
os.makedirs(os.path.dirname(save_path), exist_ok=True)

total_steps = len(lines) // batch_size

for epoch in range(epochs):
    gpt.train()
    start_time = time.time()
    pbar = tqdm(enumerate(get_batches(lines, batch_size)), total=total_steps, desc=f"Epoch {epoch+1}/{epochs}")

    for step, (neg_tensor, pos_tensor) in pbar:
        optimizer.zero_grad(set_to_none=True)
        neg_tensor = neg_tensor.to(device, non_blocking=True)
        pos_tensor = pos_tensor.to(device, non_blocking=True)

        # 5. Mixed precision forward + backward
        with autocast(device_type="cuda", dtype=torch.float16):
            neg_logprob = compute_logprob(neg_tensor)
            pos_logprob = compute_logprob(pos_tensor)
            dpo_term = beta * (pos_logprob - neg_logprob)
            loss = -F.logsigmoid(dpo_term).mean() - pos_logprob.mean() * 0.1

        # AMP-safe backward + optimizer step
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        # Update progress bar
        pbar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "lr": f"{scheduler.get_last_lr()[0]:.2e}"
        })

    # Save checkpoint (one file only)
    torch.save({
        "model_state_dict": gpt.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "model_args": gpt.config.__dict__,
        "epoch": epoch + 1,
    }, save_path)

    elapsed = time.time() - start_time
    mins, secs = divmod(int(elapsed), 60)
    print(f" Epoch {epoch+1} finished in {mins}m {secs}s — saved to {save_path}")


Using device: cuda


Epoch 1/3: 100%|██████████| 3125/3125 [12:45<00:00,  4.08it/s, loss=0.0241, lr=7.75e-05]


✅ Epoch 1 finished in 14m 2s — saved to ./dpo.pt


Epoch 2/3: 100%|██████████| 3125/3125 [12:45<00:00,  4.08it/s, loss=0.0218, lr=3.25e-05]


✅ Epoch 2 finished in 13m 38s — saved to ./dpo.pt


Epoch 3/3: 100%|██████████| 3125/3125 [12:45<00:00,  4.08it/s, loss=0.0199, lr=1.00e-05]


✅ Epoch 3 finished in 13m 14s — saved to ./dpo.pt


### Step 8: Begin testing (**students are required to complete this part!**)
Testing Process:

1. Load Fine-tuned Model:
   - Load checkpoint saved during training
   - Restore model weights and configuration
   
2. Evaluation:
   - Set model to eval mode (disables dropout, etc.)
   - Test on various math problems
   - Generate responses with controlled randomness
   
3. Expected Behavior:
   - Model should now provide correct answers
   - Responses should include explanations
   - Format: "The answer is X because..."
   
Test Set Coverage:
- Addition: 17+19=?
- Multiplication: 3*17=?
- Division: 72/4=?
- Subtraction with variable: 72-x=34,x=?
- Multiplication with variable: x*11=44,x=?

Success Criteria:
- Majority of answers should be correct
- Model should provide reasoning/explanation
- Format should match positive training examples

In [ ]:
# Load the fine-tuned model
ckpt_path = "./dpo.pt"
checkpoint = torch.load(ckpt_path, map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
gpt = GPT(gptconf).cuda()
try:
    state_dict = checkpoint['model']
except:
    state_dict = checkpoint['model_state_dict']
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
# Test
gpt.eval()
test_set = ["17+19=?", "3*17=?", "72/4=?", "72-x=34,x=?", "x*11=44,x=?", "3*17=?", "72/4=?"]
with torch.no_grad():
    for prompt in test_set:
        prompt_ids = encode(prompt)
        x = torch.tensor(prompt_ids, dtype=torch.long, device=device).unsqueeze(0)
        y, _ = gpt.generate(x, max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)
        def safe_decode(l):
          if isinstance(l, torch.Tensor):
            l = l.tolist()
            return ''.join([itos.get(i, '?') for i in l])

    generated_text = safe_decode(y[0])

    print(f"{prompt} {generated_text[len(prompt):]}")



72/4=?  The answer is 13 because 72 / 4 equals 13.


In [ ]:
# Load the fine-tuned model
ckpt_path = "./dpo.pt"
checkpoint = torch.load(ckpt_path, map_location=device)

gptconf = GPTConfig(**checkpoint["model_args"])
gpt = GPT(gptconf).to(device)

try:
    state_dict = checkpoint["model"]
except:
    state_dict = checkpoint["model_state_dict"]

unwanted_prefix = "_orig_mod."
for k in list(state_dict.keys()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)

gpt.load_state_dict(state_dict)

# Test loop
gpt.eval()
test_set = [
    "17+19=?",
    "3*17=?",
    "72/4=?",
    "72-x=34,x=?",
    "x*11=44,x=?",
    "79-72=?,x=?"
]

results = []

with torch.no_grad():
    for prompt in test_set:
        prompt_ids = encode(prompt)
        x = torch.tensor(prompt_ids, dtype=torch.long, device=device).unsqueeze(0)
        y, _ = gpt.generate(x, max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)

        # Safe decode function
        def safe_decode(l):
            if isinstance(l, torch.Tensor):
                l = l.tolist()
            return ''.join([itos.get(i, '?') for i in l])

        generated_text = safe_decode(y[0])
        answer = generated_text[len(prompt):].strip()
        print(f"{prompt} {answer}")
        results.append((prompt, answer))


17+19=? The answer is 23 because 17 + 19 equals 23.
3*17=? The answer is 42 because 3 / 77 equals 42.
72/4=? The answer is 18 because 72 / 4 equals 18.
72-x=34,x=? The answer i because 72 - a3 equals .
x*11=44,x=? The answer is 14 because 14/1 equals 14.
79-72=?,x=? Th answer is because 79 - 77 e 27 equa.
